# BiLSTM + attention (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_bilstm` (word indices + sequence lengths).

**Run modes** (set flags in the next code cell): **quick** smoke test (`QUICK_ITERATION = True`), **progress report** (`QUICK_ITERATION = False`, `PROGRESS_REPORT_MODE = True`), or **full** (both `False`).

**Metrics:** Same proposal bundle as other starter notebooks.

In [14]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [21]:
QuickSetEpochs = 8
QuickSetSamples = 75_000 
QuickSetBatchSize = 128
QuickSetMaxLen = 100
QuickSetVocab = 20_000

In [22]:
import time
import torch
import numpy as np
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = pick_device()
print("Device:", DEVICE)
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_bilstm

QUICK_ITERATION = False
PROGRESS_REPORT_MODE = True

LR = 1e-3

QUICK_MAX_TRAIN_SAMPLES = 2048
QUICK_MAX_VAL_SAMPLES = 512
QUICK_MAX_LEN = 64
QUICK_BATCH_SIZE = 128
QUICK_EPOCHS = 1
QUICK_MAX_VOCAB = 3000
QUICK_MIN_FREQ = 1
QUICK_HIDDEN = 64

PROGRESS_MAX_TRAIN_SAMPLES = QuickSetSamples # 10_000
PROGRESS_MAX_VAL_SAMPLES = 5_000 # 2_000
PROGRESS_MAX_LEN = QuickSetMaxLen # 128
PROGRESS_BATCH_SIZE = QuickSetBatchSize # 64
PROGRESS_EPOCHS = QuickSetEpochs # 2
PROGRESS_MAX_VOCAB = QuickSetVocab # 10_000
PROGRESS_MIN_FREQ = 2
PROGRESS_HIDDEN = 128

FULL_MAX_TRAIN_SAMPLES = None
FULL_MAX_VAL_SAMPLES = None
FULL_MAX_LEN = 256
FULL_BATCH_SIZE = 64
FULL_EPOCHS = 1
FULL_MAX_VOCAB = 50_000
FULL_MIN_FREQ = 2
FULL_HIDDEN = 128

if QUICK_ITERATION:
    run_mode = "quick (smoke test)"
    _train_n = QUICK_MAX_TRAIN_SAMPLES
    _val_n = QUICK_MAX_VAL_SAMPLES
    MAX_LEN = QUICK_MAX_LEN
    BATCH_SIZE = QUICK_BATCH_SIZE
    EPOCHS = QUICK_EPOCHS
    MAX_VOCAB = QUICK_MAX_VOCAB
    MIN_FREQ = QUICK_MIN_FREQ
    HIDDEN = QUICK_HIDDEN
elif PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    _train_n = PROGRESS_MAX_TRAIN_SAMPLES
    _val_n = PROGRESS_MAX_VAL_SAMPLES
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
    HIDDEN = PROGRESS_HIDDEN
else:
    run_mode = "full"
    _train_n = FULL_MAX_TRAIN_SAMPLES
    _val_n = FULL_MAX_VAL_SAMPLES
    MAX_LEN = FULL_MAX_LEN
    BATCH_SIZE = FULL_BATCH_SIZE
    EPOCHS = FULL_EPOCHS
    MAX_VOCAB = FULL_MAX_VOCAB
    MIN_FREQ = FULL_MIN_FREQ
    HIDDEN = FULL_HIDDEN

data = preprocess_for_bilstm(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)

n_train, n_val = data.X_train.shape[0], data.X_val.shape[0]
print("=" * 60)
print("CONFIG")
print("  mode:", run_mode)
print("  device:", DEVICE)
print("  train_samples:", n_train, "| val_samples:", n_val)
print("  MAX_LEN:", MAX_LEN, "| vocab_size:", vocab_size, "| HIDDEN:", HIDDEN)
print("  BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS)
print("=" * 60)

Device: cpu
CONFIG
  mode: progress report
  device: cpu
  train_samples: 75000 | val_samples: 5000
  MAX_LEN: 100 | vocab_size: 20000 | HIDDEN: 128
  BATCH_SIZE: 128 | EPOCHS: 8


In [ ]:
class AttentionBiLSTM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden: int,
        num_labels: int,
        padding_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.spatial_dropout = nn.Dropout2d(0.4)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden * 2)
        
        self.attn_net = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden * 2, num_labels)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        emb = emb.permute(0, 2, 1)
        emb = self.spatial_dropout(emb)
        emb = emb.permute(0, 2, 1)
        
        lens = lengths.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        
        out = self.layer_norm(out)
        
        scores = self.attn_net(out).squeeze(-1)
        positions = torch.arange(out.size(1), device=x.device).unsqueeze(0).expand_as(scores)
        mask = positions < lengths.unsqueeze(1).to(x.device)
        scores = scores.masked_fill(~mask, -1e9)
        
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx = (out * w).sum(dim=1)
        
        return self.fc(self.dropout(ctx))

pos_counts = torch.tensor(data.y_train.sum(axis=0))
neg_counts = len(data.y_train) - pos_counts
pos_weights = (neg_counts / (pos_counts + 1e-5)).to(DEVICE)

model = AttentionBiLSTM(vocab_size, embed_dim, HIDDEN, num_labels).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights) 
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.length_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(
    torch.tensor(data.X_val, dtype=torch.long),
    torch.tensor(data.length_val, dtype=torch.long),
    torch.tensor(data.y_val, dtype=torch.float32),
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

patience = 3
counter = 0
best_val_loss = float('inf')
best_model_state = None

t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for xb, lenb, yb in train_loader:
        xb, lenb, yb = xb.to(DEVICE), lenb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, lenb)
        loss = loss_fn(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    final_train_loss = epoch_loss / len(train_loader)
    
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for xv, lenv, yv in val_loader:
            xv, lenv, yv = xv.to(DEVICE), lenv.to(DEVICE), yv.to(DEVICE)
            val_loss_sum += loss_fn(model(xv, lenv), yv).item()
    val_loss = val_loss_sum / len(val_loader)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {final_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    scheduler.step()
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

if best_model_state:
    model.load_state_dict(best_model_state)

train_seconds = time.perf_counter() - t0
print(f"Training wall time: {train_seconds:.1f} s")
print(f"Final train loss: {final_train_loss:.4f} | val loss: {val_loss:.4f}")

C:\Users\Prana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\nn\modules\dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  return F.dropout2d(input, self.p, self.training, self.inplace)


Epoch 1/8 - Loss: 0.8933 | Val Loss: 0.5984
Epoch 2/8 - Loss: 0.6451 | Val Loss: 0.5598
Epoch 3/8 - Loss: 0.5393 | Val Loss: 0.4972
Epoch 4/8 - Loss: 0.4615 | Val Loss: 0.4695
Epoch 5/8 - Loss: 0.4414 | Val Loss: 0.4531
Epoch 6/8 - Loss: 0.4167 | Val Loss: 0.4419


In [ ]:
from sklearn.metrics import f1_score

def predict_logits_lengths(model, X_np, len_np, batch_size=512):
    model.eval()
    all_logits = []
    x = torch.tensor(X_np, dtype=torch.long)
    lens = torch.tensor(len_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE), lens[i : i + batch_size].to(DEVICE))
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)

# 1. Get raw probabilities
logits_val = predict_logits_lengths(model, data.X_val, data.length_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()

def find_best_thresholds_brute_force(y_true, y_probs, labels):
    best_thresholds = []
    print("--- Searching for Optimal Thresholds (F1 Maximization) ---")
    
    for i, label in enumerate(labels):
        y_t = y_true[:, i]
        y_p = y_probs[:, i]
        
        current_best_f1 = -1.0
        current_best_t = 0.5
        
        # --- Stage 1: Coarse Search (10ths: 0.1 to 0.9) ---
        for t in np.arange(0.1, 1.0, 0.1):
            score = f1_score(y_t, (y_p >= t).astype(int), zero_division=0)
            if score > current_best_f1:
                current_best_f1 = score
                current_best_t = t
        
        # --- Stage 2: Fine Search (100ths: +/- 0.1 around Stage 1 best) ---
        search_min = max(0.01, current_best_t - 0.1)
        search_max = min(0.99, current_best_t + 0.1)
        for t in np.arange(search_min, search_max + 0.001, 0.01):
            score = f1_score(y_t, (y_p >= t).astype(int), zero_division=0)
            if score > current_best_f1:
                current_best_f1 = score
                current_best_t = t
                
        best_thresholds.append(current_best_t)
        print(f"  {label:15} -> Best Threshold: {current_best_t:.2f} | F1: {current_best_f1:.4f}")
        
    return np.array(best_thresholds)

# 2. Execute search and apply thresholds
opt_thresholds = find_best_thresholds_brute_force(data.y_val, prob_val, LABEL_COLUMNS)
pred_val = (prob_val >= opt_thresholds).astype(np.int64)

# 3. Final Evaluation
per_label_df, summary = multilabel_evaluation_report(data.y_val, pred_val, prob_val, LABEL_COLUMNS)
print("\n--- Final Report (Optimized via Grid Search) ---")
print("Micro / macro F1:", summary)
display(per_label_df)

for name, m in per_label_confusion_matrices(data.y_val, pred_val, LABEL_COLUMNS).items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common with very few epochs)."
        " ROC-AUC may still be > 0.5. Use progress/full mode and more EPOCHS for meaningful F1."
    )

print()
print("--- Progress report (concise) ---")
print(f"  mode: {run_mode}")
print(f"  final_train_loss: {final_train_loss:.4f}")
print(f"  val_loss: {val_loss:.4f}")
print(f"  f1_micro: {summary['f1_micro']:.4f}")
print(f"  f1_macro: {summary['f1_macro']:.4f}")

--- Searching for Optimal Thresholds (F1 Maximization) ---
  toxic           -> Best Threshold: 0.81 | F1: 0.7188
  severe_toxic    -> Best Threshold: 0.95 | F1: 0.4750
  obscene         -> Best Threshold: 0.96 | F1: 0.7131
  threat          -> Best Threshold: 0.99 | F1: 0.3704
  insult          -> Best Threshold: 0.95 | F1: 0.6429
  identity_hate   -> Best Threshold: 0.98 | F1: 0.2857

--- Final Report (Optimized via Grid Search) ---
Micro / macro F1: {'f1_micro': 0.6612096407457936, 'f1_macro': 0.5343114196111128}


,label,precision,recall,f1,roc_auc
0,toxic,0.698000,0.740977,0.718847,0.955100
1,severe_toxic,0.358491,0.703704,0.475000,0.986783
2,obscene,0.797170,0.645038,0.713080,0.965874
3,threat,0.277778,0.555556,0.370370,0.994434
4,insult,0.653846,0.632231,0.642857,0.959818
5,identity_hate,0.254902,0.325000,0.285714,0.956568


toxic 
 [[4378  151]
 [ 122  349]]
severe_toxic 
 [[4878   68]
 [  16   38]]
obscene 
 [[4695   43]
 [  93  169]]
threat 
 [[4978   13]
 [   4    5]]
insult 
 [[4677   81]
 [  89  153]]
identity_hate 
 [[4922   38]
 [  27   13]]

--- Proposal summary (record for report / comparison) ---
  device: cpu
  training_time_s: 4641.05
  num_parameters: 1770599

--- Progress report (concise) ---
  mode: progress report
  final_train_loss: 0.2659
  val_loss: 0.4519
  f1_micro: 0.6612
  f1_macro: 0.5343


## Notes

- Increase `EPOCHS` and add validation-based **early stopping** for the final report.
- Tune **class weights** in `BCEWithLogitsLoss` (`pos_weight`) for rare heads (`threat`, `identity_hate`).